In [ ]:
"""
   Copyright (C) Sergei Vostrikov (Sergio5714), 2026
   
   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at
       http://www.apache.org/licenses/LICENSE-2.0
   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.

   SPDX-License-Identifier: Apache-2.0
"""

# WULPUS Pro Max firmware update

## ESP32-C6

Close the acquisition GUI and all serial monitors. Select the release ZIP and the ESP32-C6 COM port, acknowledge the power warning, and click **Flash ESP32**. If automatic bootloader entry fails, hold **BOOT**, tap **RESET**, release **BOOT**, and retry.

In [ ]:
from wulpus.esp32_update import Esp32UpdateGui

esp32_updater = Esp32UpdateGui()
esp32_updater.show()

## MSP430FR5043 (manual separate operation)

After the ESP32 is running normally, use this panel to upload the released MSP430 ZIP through the ESP32. TI-TXT, Intel HEX, and `.mspfw` files remain supported for development. Disconnect any MSP-FET and follow the [firmware update guide](../docs/firmware_update_guide.md).

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from wulpus.usb_cdc_link import WulpusProUsbCdcLink
from wulpus.msp430_update import MSP430Updater

msp_link = WulpusProUsbCdcLink()
msp_port = widgets.Dropdown(description='COM port:', options=[])
msp_refresh = widgets.Button(description='Refresh ports', icon='refresh')
msp_open = widgets.Button(description='Open', button_style='info')
msp_disconnect = widgets.Button(description='Disconnect', disabled=True)
msp_status = widgets.HTML(value='<b>Status:</b> Not connected')
msp_panel = widgets.Output()

def refresh_msp_ports(_=None):
    devices = msp_link.get_available()
    msp_port.options = [(str(device), device) for device in devices]
    msp_open.disabled = not devices or msp_link.connected

def open_msp_port(_):
    if msp_port.value is None or not msp_link.open(msp_port.value):
        msp_status.value = '<b>Status:</b> Failed to open device'
        return
    msp_status.value = f'<b>Status:</b> Connected to {msp_port.value.device}'
    msp_open.disabled = True
    msp_disconnect.disabled = False
    with msp_panel:
        MSP430Updater(msp_link).show_widget()

def disconnect_msp_port(_):
    if msp_link.connected:
        msp_link.close()
    msp_panel.clear_output()
    msp_status.value = '<b>Status:</b> Disconnected'
    msp_disconnect.disabled = True
    refresh_msp_ports()

msp_refresh.on_click(refresh_msp_ports)
msp_open.on_click(open_msp_port)
msp_disconnect.on_click(disconnect_msp_port)
display(widgets.VBox([widgets.HBox([msp_port, msp_refresh, msp_open, msp_disconnect]), msp_status, msp_panel]))
refresh_msp_ports()